## 0. Setup

Import all the libraries needed for the project. No installation required — everything here is already available on Kaggle.

In [1]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from itertools import combinations

RANDOM_STATE = 42
sns.set_style("whitegrid")

## 1. Data Understanding

Load the dataset and take a first look at its shape, columns, and any missing values.

In [2]:
candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("CSV files found:")
for c in candidates:
    print(" -", c)

CSV_PATH = candidates[0] if candidates else "shopping_behaviour.csv"
print("\nUsing:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

CSV files found:
 - /kaggle/input/datasets/saadaliyaseen/shopping-behaviour-dataset/shopping_behavior_updated (1).csv

Using: /kaggle/input/datasets/saadaliyaseen/shopping-behaviour-dataset/shopping_behavior_updated (1).csv
(3900, 18)


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


### 1.1 Check for missing values

In [3]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values per column:
Series([], dtype: int64)


### 1.2 Define the target variable

We predict `Subscription Status` (Yes/No) — a genuine binary column in this dataset — rather than a general purchase flag, since every row here is already a completed purchase.

In [4]:
TARGET_COL = "Subscription Status"

y_raw = df[TARGET_COL]
print(y_raw.value_counts())

Subscription Status
No     2847
Yes    1053
Name: count, dtype: int64


## 2. Data Preparation

Separate features from the target, and prepare text columns for encoding.

In [5]:
id_like = ["Customer ID"]
feature_cols = [c for c in df.columns if c not in id_like + [TARGET_COL]]

numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)

X = df[feature_cols].copy()
y = LabelEncoder().fit_transform(df[TARGET_COL])

Numeric features: ['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']
Categorical features: ['Gender', 'Item Purchased', 'Category', 'Location', 'Size', 'Color', 'Season', 'Shipping Type', 'Discount Applied', 'Promo Code Used', 'Payment Method', 'Frequency of Purchases']


### 2.1 Select features (excluding near-perfect predictors)

Gender, Discount Applied, and Promo Code Used are excluded from the model's inputs — the crosstab showed they almost perfectly determine Subscription Status on their own, which would make the classification task trivial rather than a meaningful test of the models. They're noted as a finding, not used as inputs.

In [6]:
id_like = ["Customer ID"]
leak_like = ["Gender", "Discount Applied", "Promo Code Used"]  # near-perfect predictors, excluded on purpose

feature_cols = [c for c in df.columns if c not in id_like + leak_like + [TARGET_COL]]

numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print("Excluded (ID):", id_like)
print("Excluded (near-perfect predictors):", leak_like)
print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)

X = df[feature_cols].copy()
y = LabelEncoder().fit_transform(df[TARGET_COL])

Excluded (ID): ['Customer ID']
Excluded (near-perfect predictors): ['Gender', 'Discount Applied', 'Promo Code Used']
Numeric features: ['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']
Categorical features: ['Item Purchased', 'Category', 'Location', 'Size', 'Color', 'Season', 'Shipping Type', 'Payment Method', 'Frequency of Purchases']


In [7]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
])

X_processed = preprocessor.fit_transform(X)
feature_names = (numeric_cols +
                  list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)))
X_processed_df = pd.DataFrame(X_processed, columns=feature_names)
X_processed_df.shape

(3900, 135)